# Dashboard nachvollziehen

Dieselbe Auswahl wie in der Weboberfläche. Die Schalter, die drei Kreise, der Trendpfeil und die Wortwolke rufen Funktionen aus `dashboard.py` auf.

Änderungen an Städten, Medien und an der Blasen-Quelle stehen oben in `dashboard.py` im Block `KONFIGURATION`. Danach das Skript neu starten.

Die Weboberfläche selbst startet im Terminal, im Ordner `Hackathon-KJRJ`:

```bash
python dashboard.py
```

Im Browser: [http://127.0.0.1:8765](http://127.0.0.1:8765)

In [ ]:
from pathlib import Path
import sys

ordner = Path.cwd().resolve()
for kandidat in (ordner, ordner.parent, ordner / "Hackathon-KJRJ", ordner.parent / "Hackathon-KJRJ"):
    if (kandidat / "dashboard.py").is_file():
        ordner = kandidat
        break
sys.path.insert(0, str(ordner))

import dashboard as dash

print("Daten:", dash.DATEN_DATEI)
print("Blasen-Quelle:", dash.BLASEN_QUELLE)
print("Städte-Buttons:", [eintrag["id"] for eintrag in dash.STAEDTE_BUTTONS])
print("Projekt -> Stadt:")
for projekt, stadt in dash.STADT_AUS_PROJEKT.items():
    print(f"  {projekt} -> {stadt}")
print("Kanal -> Schalter:")
for kanal, schalter in dash.MEDIEN_AUS_KANAL.items():
    print(f"  {kanal} -> {schalter}")

## Beiträge laden

`lade_beitraege` liest `daten/01_gefiltert/clean_data.json` und hängt die Spalten an, die das Dashboard filtert:

- `stadt` aus `STADT_AUS_PROJEKT` (IPAI zählt zu Heilbronn)
- `medium` aus dem Kanal: Nachrichten werden Zeitungen, Social Media bleibt Social Media
- `blase` ist Zustimmung, Neutral oder Ablehnung
- `haltung` kommt aus den Textmustern, `vendor_tonlage` aus dem Feld `sentiment`
- `woerter` sind die Begriffe für die Wortwolke, ohne Paywall-Hinweis der Stimme

In [ ]:
beitraege = dash.lade_beitraege()
print(f"{len(beitraege)} Beiträge")
print("Von", beitraege["datum"].min().date(), "bis", beitraege["datum"].max().date())
print("\nStadt:")
print(beitraege["stadt"].value_counts().to_string())
print("\nMedium:")
print(beitraege["medium"].value_counts(dropna=False).to_string())
print("\nBlase:")
print(beitraege["blase"].value_counts(dropna=False).to_string())
dash.ueberblick()

## Dieselbe Auswahl wie die Schalter

Die drei Zeilen unten entsprechen den Buttons und dem Zeitstrahl.
`Gesamt` lässt die Stadt weg. `von` und `bis` sind inklusive.
Liegt `von` auf `None`, nimmt das Dashboard den ganzen Zeitraum der aktuellen Stadt und des aktuellen Mediums.

In [ ]:
STADT = "Heilbronn"       # Gesamt, Heilbronn, Bonn, Paris
MEDIUM = "Zeitungen"      # Zeitungen, Social Media
VON = "2025-06-01"
BIS = "2025-12-31"

sicht = dash.auswertung(stadt=STADT, medium=MEDIUM, von=VON, bis=BIS)

print(sicht["stadt"], "|", sicht["medium"])
print("Zeitstrahl:", sicht["von"], "bis", sicht["bis"])
print("Datenpunkte:", sicht["datenpunkte"])
print("\nKreise:")
for name in dash.BLASEN:
    anzahl = sicht["blasen"]["anzahl"][name]
    anteil = sicht["blasen"]["anteil"][name]
    print(f"  {name:<12} {anzahl:4d}   {anteil:.0%}")
print("ohne Einordnung:", sicht["blasen"]["ohne_blase"])
print("\nTrend:", sicht["trend"]["richtung"])
print(sicht["trend"]["satz"])
print("\nZehn häufigste Begriffe:")
for eintrag in sicht["woerter"]:
    print(f"  {eintrag['anzahl']:4d}  {eintrag['wort']}")

## Die Beiträge hinter der Zahl

Ein Klick auf die Datenpunkte in der Oberfläche zeigt dieselben Zeilen: Datum, Projekt, Tonlage, Haltung im Text, Titel.

In [ ]:
ausschnitt = dash.filtere(beitraege, stadt=STADT, medium=MEDIUM, von=VON, bis=BIS)
spalten = ["datum", "projekt_label", "medium", "blase", "vendor_tonlage", "haltung", "title", "source_name"]
ausschnitt.sort_values("datum", ascending=False)[spalten].head(12)

## Was die Oberfläche aufruft

| Element | Funktion |
| --- | --- |
| Stadt-Buttons und Medien-Schalter | `filtere` |
| Zeitstrahl | `tagesreihe` auf der Auswahl ohne Datumsfenster |
| Drei Kreise | `blasen_zaehlen` |
| Pfeil daneben | `trend`, Vergleich mit dem direkt davor liegenden Fenster gleicher Länge |
| Zahl rechts | Länge des Ausschnitts, Klick lädt `beitraege_liste` |
| Wortwolke | `haeufigste_woerter` |

`auswertung` bündelt genau das Paket, das der Browser unter `/api/auswertung` bekommt.